In [5]:
import pandas as pd

file_path = "news_dataset.csv"
df = pd.read_csv(file_path, encoding='latin1')

df = df[["text"]]
df = df.dropna()
#df = df.sample(n=5000, random_state=42)

pd.set_option('display.max_colwidth', None)
print("Successful import")
print(f"df_main shape:        {df.shape}")

Successful import
df_main shape:        (11096, 1)


In [7]:
def convert_to_lowercase(text):
    return text.lower()
df["lowercased"] = df["text"].apply(convert_to_lowercase)

pd.set_option('display.max_colwidth', None)
print(df["lowercased"])
print(f"df_main shape:        {df.shape}")

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [13]:
import re

def clean_text(text):
    # Remove URLs
    text = re.sub(r'https\S+|www\S+', '', text)
    # Remove newlines and replace with a space
    text = text.replace('\n', ' ')
    # Optional: Remove extra whitespace created by the replacements
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["urls_removed"] = df["lowercased"].apply(clean_text)

In [14]:
from bs4 import BeautifulSoup

def remove_html_tags(text):
    return BeautifulSoup(text, "html.parser").get_text()
df["html_removed"] = df["urls_removed"].apply(remove_html_tags)

pd.set_option('display.max_colwidth', None)
print(df["html_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [16]:
# Removal of emojis (if any) 
import emoji 
# replace emoji with '' 
def remove_emojis(text): 
    return emoji.replace_emoji(text, replace='') 
df["emojis_removed"] = df["html_removed"].apply(remove_emojis) 
# Display column content without truncation 

pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["emojis_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [24]:
# Replace internet slang/chat words 
# Dictionary of slang words and their replacements 
slang_dict = { 
    "tbh": "to be honest", 
    "omg": "oh my god", 
    "lol": "laugh out loud", 
    "idk": "I don't know", 
    "brb": "be right back", 
    "btw": "by the way", 
    "imo": "in my opinion", 
    "smh": "shaking my head", 
    "fyi": "for your information", 
    "np": "no problem",
    "ikr": "I know right", 
    "asap": "as soon as possible", 
    "bff": "best friend forever", 
    "gg": "good game", 
    "hmu": "hit me up", 
    "rofl": "rolling on the floor laughing" ,
    "yrs" :"years"
}
# Function to replace slang words 
def replace_slang(text): 
    # Create a list of escaped slang words 
    escaped_slang_words = []  # Empty list to store escaped slang words
    for word in slang_dict.keys(): 
        escaped_word = re.escape(word)  # Ensure special characters are escaped 
        escaped_slang_words.append(escaped_word)  # Add to list
    # Join the words using '|' 
    slang_pattern = r'\b(' + '|'.join(escaped_slang_words) + r')\b'
    # Define a replacement function 
    def replace_match(match): 
        slang_word = match.group(0)  # Extract matched slang word 
        return slang_dict[slang_word.lower()]  # Replace with full form
    # Use regex to replace slang words with full forms 
    replaced_text = re.sub(slang_pattern, replace_match, text, flags=re.IGNORECASE)
    return replaced_text 

# Apply the function to the column 
df["slangs_replaced"] = df["emojis_removed"].apply(replace_slang)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["slangs_replaced"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [25]:
# Replace Contractions 
contractions_dict = { 
    "wasn't": "was not", 
    "isn't": "is not", 
    "aren't": "are not", 
    "weren't": "were not", 
    "doesn't": "does not", 
    "don't": "do not", 
    "didn't": "did not", 
    "can't": "cannot", 
    "couldn't": "could not", 
    "shouldn't": "should not", 
    "wouldn't": "would not", 
    "won't": "will not", 
    "haven't": "have not", 
    "hasn't": "has not", 
    "hadn't": "had not", 
    "i'm": "i am", 
    "you're": "you are", 
    "he's": "he is", 
    "she's": "she is", 
    "it's": "it is", 
    "we're": "we are", 
    "they're": "they are", 
    "i've": "i have", 
    "you've": "you have", 
    "we've": "we have", 
    "they've": "they have", 
    "i'd": "i would", 
    "you'd": "you would", 
    "he'd": "he would", 
    "she'd": "she would", 
    "we'd": "we would", 
    "they'd": "they would", 
    "i'll": "i will",
    "you'll": "you will", 
    "he'll": "he will", 
    "she'll": "she will", 
    "we'll": "we will", 
    "they'll": "they will", 
    "let's": "let us", 
    "that's": "that is", 
    "who's": "who is", 
    "what's": "what is", 
    "where's": "where is", 
    "when's": "when is", 
    "why's": "why is" 
}

# Build the regex pattern for contractions 
escaped_contractions = []  # List to store escaped contractions

for contraction in contractions_dict.keys(): 
    escaped_contraction = re.escape(contraction)  # Escape special characters (e.g., apostrophes)
    escaped_contractions.append(escaped_contraction)  # Add to list

# Join the escaped contractions with '|' 
joined_contractions = "|".join(escaped_contractions) 

# Create a regex pattern with word boundaries (\b) 
contractions_pattern = r'\b(' + joined_contractions + r')\b' 

# Compile the regex 
compiled_pattern = re.compile(contractions_pattern, flags=re.IGNORECASE)

# Define a function to replace contractions 
def replace_contractions(text): 
    # Function to handle each match found
    def replace_match(match): 
        matched_word = match.group(0)  # Extract matched contraction 
        lower_matched_word = matched_word.lower()  # Convert to lowercase
        expanded_form = contractions_dict[lower_matched_word]  #Get full form from dictionary
        return expanded_form #Return the expanded form
    
    #Apply regex substitution
    expanded_text = compiled_pattern.sub(replace_match, text)

    return expanded_text #Return modified text

# Apply the function to a DataFrame column 
df["contractions_replaced"] = df["slangs_replaced"].apply(replace_contractions)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["contractions_replaced"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [26]:
# Remove punctuations and special characters 
import string

# Function to remove punctuation 
def remove_punctuation(text): 
    return text.translate(str.maketrans('', '', string.punctuation))

# Apply the function to the column 
df["punctuations_removed"] = df["contractions_replaced"].apply(remove_punctuation)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["punctuations_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [27]:
# Remove numbers 
def remove_numbers(text): 
    return re.sub(r'\d+', '', text)  # Removes all numeric characters

# Apply the function to the column 
df["numbers_removed"] = df["punctuations_removed"].apply(remove_numbers)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["numbers_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [28]:
# Remove stopwords 
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already downloaded 
nltk.download('stopwords')

# Define stopwords list 
stop_words = set(stopwords.words('english'))

# Function to remove stopwords 
def remove_stopwords(text):
    words = text.split()  # Split text into words 
    filtered_words = []  # Create an empty list to store words after stopword removal

    for word in words:  # Loop through each word in the list of words
        lower_word = word.lower() #Convert the word to lowercase for uniform comparison
        if lower_word not in stop_words:  # Check if the lowercase word is NOT in the stopwords list
            filtered_words.append(word)  # If it's not a stopword, add it to the filtered list
    return " ".join(filtered_words)  # Join words back into a sentence

# Apply the function to the column 
df["stopwords_removed"] = df["numbers_removed"].apply(remove_stopwords)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["stopwords_removed"])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kamin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [29]:
# Stemming - reduces words to their base root by chopping off suffixes 
from nltk.stem import PorterStemmer

# Initialize the stemmer 
stemmer = PorterStemmer() 

# Function to apply stemming 
def stem_text(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]  # Apply stemming
    return " ".join(stemmed_words)

# Apply the function 
df["stemmed_words"] = df["stopwords_removed"].apply(stem_text)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["stemmed_words"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
import nltk

# Lemmatization - reduces words to their base dictionary form (lemma)
from nltk.stem import WordNetLemmatizer 
from nltk.corpus import wordnet 
from nltk.tokenize import word_tokenize 
from nltk import pos_tag

# Initialize the lemmatizer 
lemmatizer = WordNetLemmatizer()

# Function to map NLTK POS tags to WordNet POS tags 
def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'):  # Adjective
         return wordnet.ADJ
    elif nltk_tag.startswith('V'):  # Verb 
        return wordnet.VERB
    elif nltk_tag.startswith('N'): #Noun
        return wordnet.NOUN
    elif nltk_tag.startswith('R'): #Adverb
        return wordnet.ADV
    else:
        return wordnet.NOUN #Default to noun

#Function to lemmatize text with POS tagging
def lemmatize_text(text):
    if not isinstance(text,str): #Ensure input is string
        return ""
    words = word_tokenize(text) #Tokenize text into words
    pos_tags = pos_tag(words) #get POS tags

    #Lemmatize each word with its correct POS tag
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]

    return " ".join(lemmatized_words)  # Join words back into a sentence

# Apply the function to the column 
df["lemmatized"] = df["stopwords_removed"].apply(lemmatize_text)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["lemmatized"])


0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [37]:
df.to_csv("Processed_News.csv", index=False)
print(df.columns)

Index(['text', 'lowercased', 'urls_removed', 'html_removed', 'emojis_removed',
       'slangs_replaced', 'contractions_replaced', 'punctuations_removed',
       'numbers_removed', 'stopwords_removed', 'stemmed_words', 'lemmatized'],
      dtype='object')


In [54]:
preprocessed_documents = df["lemmatized"].apply(
    lambda x: [
        word for word in x.split()
        if len(word) > 2                      # remove short words
        and word.isalpha()                   # remove weird tokens
        and word not in ['would','one','get','say','think','know']  # remove common fillers
    ]
).tolist()
documents = df["text"].tolist()  # original text

In [55]:
from gensim import corpora

dictionary = corpora.Dictionary(preprocessed_documents)

# Better settings for large dataset (11k rows)
dictionary.filter_extremes(no_below=20, no_above=0.4)

corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

In [56]:
from gensim.models import LdaModel

num_topics = 5

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    passes=20,
    random_state=42
)

In [57]:
from gensim.models import CoherenceModel

coherence_model = CoherenceModel(
    model=lda_model,
    texts=preprocessed_documents,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print("Coherence Score:", coherence_score)
print()

Coherence Score: 0.4985528498262345



In [61]:
import pandas as pd

article_labels = []

for doc in preprocessed_documents:
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    article_labels.append(dominant_topic)

df_result = pd.DataFrame({
    "Article": documents,
    "Topic": article_labels
})

print("Table with News and Topic:")
print(df_result.head())  # only show first few
print()

Table with News and Topic:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [59]:
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()

Top terms for Topic #0:
['game', 'year', 'team', 'good', 'like', 'well', 'play', 'time', 'make', 'new']

Top terms for Topic #1:
['people', 'make', 'god', 'see', 'like', 'thing', 'believe', 'even', 'come', 'time']

Top terms for Topic #2:
['use', 'key', 'file', 'system', 'chip', 'program', 'bit', 'also', 'work', 'need']

Top terms for Topic #3:
['government', 'state', 'president', 'people', 'armenian', 'law', 'year', 'right', 'american', 'new']



In [60]:
print("Top Terms for Each Topic:")

for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]

    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "game" (weight: 0.010)
- "year" (weight: 0.010)
- "team" (weight: 0.008)
- "good" (weight: 0.008)
- "like" (weight: 0.007)
- "well" (weight: 0.007)
- "play" (weight: 0.006)
- "time" (weight: 0.006)
- "make" (weight: 0.006)
- "new" (weight: 0.005)

Topic 1:
- "people" (weight: 0.013)
- "make" (weight: 0.009)
- "god" (weight: 0.007)
- "see" (weight: 0.007)
- "like" (weight: 0.007)
- "thing" (weight: 0.006)
- "believe" (weight: 0.006)
- "even" (weight: 0.006)
- "come" (weight: 0.006)
- "time" (weight: 0.006)

Topic 2:
- "use" (weight: 0.021)
- "key" (weight: 0.012)
- "file" (weight: 0.010)
- "system" (weight: 0.009)
- "chip" (weight: 0.007)
- "program" (weight: 0.006)
- "bit" (weight: 0.006)
- "also" (weight: 0.005)
- "work" (weight: 0.005)
- "need" (weight: 0.005)

Topic 3:
- "government" (weight: 0.012)
- "state" (weight: 0.008)
- "president" (weight: 0.007)
- "people" (weight: 0.006)
- "armenian" (weight: 0.006)
- "law" (weight: 0.005)
- "year" (wei

Kaminy Varma (IS01083876)

Topic 0: Sports
Topic 1: Religion
Topic 2:Technology
Topic 3:Politics

The coherence score of 0.4986 indicates a moderate to good level of topical concictency, suggesting that the model was able to identify word clusters that represent meaningful themes within the dataset. However, improvements can be made as a score closer to 1 is more significant.